# 1. Classification

## Question 1

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, confusion_matrix, ConfusionMatrixDisplay,
    f1_score, classification_report, mean_squared_error, r2_score
)
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import make_pipeline
from statistics import mean, stdev
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import PolynomialFeatures

import warnings
warnings.filterwarnings("ignore")

In [ ]:
data = pd.read_csv("data.csv", delimiter=";")
data.head()


In [ ]:
data['Target'] = data['Target'].replace({'Dropout': 0, 'Graduate': 1, 'Enrolled': 1})
data_all = data.copy()
data_target = data['Target']
data = data.drop('Target', axis=1)

## Question 2

In [ ]:
# on normalise les valeurs de chaque attribut (tous sont de type numérique)
scaler = StandardScaler()
scaled_data = scaler.fit_transform(data)
scaled_data = pd.DataFrame(scaled_data, columns=data.columns)

In [ ]:
correlation_matrix = scaled_data.corr()

plt.figure(figsize=(18, 14))
mask = np.triu(np.ones_like(correlation_matrix, dtype=bool))
sns.heatmap(correlation_matrix, mask=mask, cmap='viridis', center=0, vmin=-1, vmax=1,
            linewidths=0.3, annot=False)
plt.title('Matrice de corrélation des features', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
threshold = 0.80
high_correlation_matrix = []
for i in range(len(correlation_matrix.columns)):
    for j in range(i):
        if correlation_matrix.iloc[i, j] > threshold:
            high_correlation_matrix.append((correlation_matrix.columns[i], correlation_matrix.columns[j], correlation_matrix.iloc[i, j]))

if high_correlation_matrix:
    print(f"Features très semblables (r > {threshold} )")
    for a, b, r in sorted(high_correlation_matrix, key=lambda x: abs(x[2]), reverse=True):
        print(f'  {a} ~ {b}  r={r:.3f}')

In [ ]:
data_all_plot = data_all.copy()
data_all_plot["Target"] = data_all_plot["Target"].map({0: "Dropout", 1: "Graduate/Enrolled"})

for attribute in data.columns:
    plt.figure(figsize=(8, 4))
    sns.histplot(
        data=data_all_plot,
        x=attribute,
        hue="Target",
        hue_order=["Dropout", "Graduate/Enrolled"]  # ordre fixe de la légende
    )
    plt.title(f'Distribution de {attribute} par classe')
    plt.xlabel(attribute)
    plt.ylabel('Fréquence')
    plt.tight_layout()
    plt.show()



## Interprétation des résultats

##### Certains attributs ne semble pas être des facteurs déterminants puisqu'ils ont la même densité pour les dropout que les autres. Ces attributs sont principalement :
-GDP; Inflation Rate;

##### Certains attributs semblent être des facteurs déterminants
- Age (aux alentours de 20 réside toute la concentration de dropout)
- Scholarship (beaucoup plus de dropout chez ceux en possédant 1)
- Gender (plus de dropout chez 1 -> male)
- Displaced (plus de dropout chez 0 -> non displaced)
- Debtor (plus de dropout chez 1 -> debtor)
- Tuition fees (moins de dropout chez ceux qui payent des frais de scolarité)


## Question 3

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, random_state=42)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier()
rf.fit(X_train, y_train)
print(f"Random Forest initial accuracy: train {rf.score(X_train, y_train)} / test {rf.score(X_test, y_test)}")

## Question 4

In [ ]:
rf = RandomForestClassifier()
rf.fit(X_train, y_train)
print(f"Random Forest initial accuracy (random_state): train {rf.score(X_train, y_train)} / test {rf.score(X_test, y_test)}")

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

rf_accuracies = []
gb_accuracies = []
seed_range = [10, 20, 30, 40, 50, 60, 70, 80, 90]

for seed in seed_range:
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, random_state=seed)
    # Random Forest
    rf_tmp = RandomForestClassifier()
    rf_tmp.fit(X_train, y_train)
    rf_accuracies.append(rf_tmp.score(X_test, y_test))

    # Gradient Boosting
    gb_tmp = GradientBoostingClassifier()
    gb_tmp.fit(X_train, y_train)
    gb_accuracies.append(gb_tmp.score(X_test, y_test))

import matplotlib.pyplot as plt

plt.plot(seed_range, rf_accuracies, label='Random Forest', marker='o')
plt.plot(seed_range, gb_accuracies, label='Gradient Boosting', marker='s')
plt.xlabel('Random State Seed')
plt.ylabel('Accuracy')
plt.legend()
plt.show()

## Question 5

## Question 6

In [ ]:

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.99, random_state=seed)

# Gradient Boosting
gb_tmp = GradientBoostingClassifier()
gb_tmp.fit(X_train, y_train)
print("Gradient Boosting Accuracy with 1% training data:", gb_tmp.score(X_test, y_test))

TODO

# 2. Regression

## Question 7

In [ ]:
measurements = pd.read_csv("measurements.csv", delimiter=',')

col_x, col_y = measurements.columns[0], measurements.columns[1]

X = measurements[[col_x]].values
y = measurements[col_y].values

lin_model = LinearRegression()
lin_model.fit(X, y)

y_pred_lin = lin_model.predict(X)

mse_lin = mean_squared_error(y, y_pred_lin)
r2_lin  = r2_score(y, y_pred_lin)

print(f'Régression linéaire')
print(f'Coefficient : {lin_model.coef_[0]:.4f}')
print(f'Intercept : {lin_model.intercept_:.4f}')
print(f'MSE : {mse_lin:.4f}')
print(f'R²: {r2_lin}')

## Interprétation des résultats

##### MSE
- L'erreur absolue moyenne est relativement proche de 0 (0.49), ce qui veut dire qu'en moyenne les points sont proches de l'estimation linéaire faite.

#### R²
- Le coefficient de regression linéaire est proche de 1 (0.874), donc c'est un bon modèle

## Question 8

In [ ]:
degrees = [2, 8, 25]
colors  = ['seagreen', 'darkorange', 'purple']

results = []
x_plot = np.linspace(X.min(), X.max(), 300).reshape(-1, 1)

fig, axes = plt.subplots(1, len(degrees), figsize=(16, 5), sharey=False)

for ax, deg, color in zip(axes, degrees, colors):
    poly_model = make_pipeline(
        PolynomialFeatures(degree=deg, include_bias=False),
        LinearRegression()
    )
    poly_model.fit(X, y)

    y_pred = poly_model.predict(X)
    mse    = mean_squared_error(y, y_pred)
    r2     = r2_score(y, y_pred)
    results.append({'Degré': deg, 'MSE': mse, 'R²': r2})

    y_curve = poly_model.predict(x_plot)

    ax.scatter(X, y, color='steelblue', alpha=0.6, edgecolors='white', s=25, linewidths=0.3)
    ax.plot(x_plot, y_curve, color=color, linewidth=2)
    ax.set_title(f'Degré {deg}\nMSE={mse:.2f}  R²={r2:.4f}', fontsize=11)
    ax.set_xlabel(col_x)
    ax.set_ylabel(col_y)

plt.suptitle('Régression polynomiale - comparaison des degrés', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

df_results = pd.DataFrame([{'Degré': 1, 'MSE': mse_lin, 'R²': r2_lin}] + results)
print(df_results.to_string(index=False))

In [ ]:
all_degrees = [1, 2, 8, 25]
mse_vals = df_results['MSE'].values
r2_vals  = df_results['R²'].values

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

ax1.plot(all_degrees, mse_vals, marker='o', color='tomato')
ax1.set_xlabel('Degré du polynôme')
ax1.set_ylabel('MSE (sur le train)')
ax1.set_title('MSE en fonction du degré')
ax1.set_xticks(all_degrees)

ax2.plot(all_degrees, r2_vals, marker='o', color='steelblue')
ax2.set_xlabel('Degré du polynôme')
ax2.set_ylabel('R² (sur le train)')
ax2.set_title('R² en fonction du degré')
ax2.set_xticks(all_degrees)

plt.tight_layout()
plt.show()